# Lab: K-Means Clustering

## 1. Học có giám sát vs không giám sát

Đến giờ ta đã làm **học có giám sát** — input có nhãn, model học để dự đoán nhãn cho input mới.

**Học không giám sát** (unsupervised): chỉ có input, không có nhãn. Mục tiêu: *khám phá cấu trúc* trong dữ liệu. Một bài toán điển hình là **clustering** — chia dữ liệu thành các cụm (group) tự nhiên.

Ví dụ ứng dụng:
- Phân khúc khách hàng theo hành vi mua sắm.
- Nhóm bài viết theo chủ đề (không cần biết trước có những chủ đề gì).
- Compress ảnh: quantize 16M màu thành 16 màu chính.

## 2. Ý tưởng K-Means

Cho $K$ (số cụm), K-Means tìm $K$ "tâm" (centroid) sao cho tổng khoảng cách bình phương từ mỗi điểm đến tâm cụm gần nhất là **nhỏ nhất**.

$$
L = \sum_{k=1}^{K} \sum_{x \in C_k} \|x - \mu_k\|^2
$$

Với $\mu_k$ là tâm của cụm $k$, $C_k$ là tập điểm thuộc cụm $k$.

## 3. Thuật toán Lloyd (kinh điển)

1. **Khởi tạo**: chọn $K$ tâm ngẫu nhiên.
2. **Bước Assign**: gán mỗi điểm vào cụm có tâm gần nhất (Euclidean distance).
3. **Bước Update**: cập nhật mỗi tâm = trung bình các điểm trong cụm.
4. Lặp lại 2-3 đến khi tâm không đổi (hoặc đạt max_iter).

Mỗi vòng đảm bảo loss giảm → thuật toán hội tụ. Nhưng **không đảm bảo cực tiểu toàn cục** — chạy lại với khởi tạo khác có thể ra kết quả khác.

## 4. Bốn vấn đề thực tế

### 4.1. Chọn $K$ thế nào?
Hai phương pháp phổ biến:
- **Elbow method**: vẽ inertia (= tổng bình phương khoảng cách từ mỗi điểm tới tâm cụm của nó, *within-cluster SSE*) theo $K$. Chọn $K$ tại "khuỷu tay" — chỗ inertia bắt đầu giảm chậm.
- **Silhouette score**: đo độ "tách" giữa các cụm. Cao = tốt.

### 4.2. Khởi tạo kém → kết quả tệ
Random init có thể đẩy thuật toán vào local minimum. **K-Means++** (mặc định trong sklearn) chọn tâm thông minh hơn → ổn định.

### 4.3. Phải scale feature
K-Means dùng Euclidean distance → feature scale lớn lấn át. Phải `StandardScaler` trước khi clustering.

### 4.4. Chỉ phù hợp với cụm tròn
K-Means giả định cụm dạng *tròn, kích thước tương đồng*. Với cụm cong (như hai trăng lưỡi liềm) hoặc kích cỡ rất khác nhau, K-Means thất bại — dùng **DBSCAN** hoặc **Spectral Clustering** thay thế.

### 3.1. Xem thuật toán Lloyd chạy từng bước

Đọc 4 dòng mô tả thuật toán thì dễ, nhưng phải *nhìn* mới thấy được điều quan trọng: **tâm cụm bị các điểm kéo về phía chúng, và các điểm lại đổi phe theo tâm** — hai bước này giằng co nhau cho tới khi cả hai cùng đứng yên.

![Thuật toán Lloyd từng bước: khởi tạo, assign, update, hội tụ](images/01_lloyd_tung_buoc.png)

*Sáu trạng thái liên tiếp của Lloyd trên cùng một bộ dữ liệu. Ba tâm được đặt cố ý lệch hẳn về một góc; sau đúng 3 vòng lặp chúng đã tìm đúng ba cụm. Con số ở góc dưới trái là inertia — nó không bao giờ tăng sau bất kỳ nửa bước nào và đi từ 14466.7 xuống 332.5. (Riêng panel "Vòng 1 — ASSIGN" giữ nguyên 14466.7 vì ở Bước 0 inertia đã được tính theo chính cách gán "tâm gần nhất" đó; bước ASSIGN đầu tiên chỉ tô màu lên.)*

Hai điều cần để ý:

1. **Bước ASSIGN không làm tâm dịch chuyển**, nó chỉ tô lại màu cho các điểm. **Bước UPDATE không đổi màu điểm nào**, nó chỉ kéo tâm về trọng tâm của màu đó.
2. Ở panel "Vòng 1 — UPDATE", tâm đỏ vẫn còn lơ lửng ở trên: cụm đỏ lúc đó **rỗng**. sklearn xử lý cụm rỗng bằng cách gán lại tâm đó cho điểm xa nhất; cài đặt `MyKMeans` trong notebook này giữ nguyên tâm cũ (xem dòng `if (labels == k).any() else centers[k]`).

### 3.2. Vì sao Lloyd CHẮC CHẮN hội tụ — chứng minh 5 dòng

Viết lại hàm mục tiêu cho tường minh, với $r_{ik}\in\{0,1\}$ là biến chỉ báo "điểm $x_i$ thuộc cụm $k$":

$$
L(\mathbf{r}, \boldsymbol{\mu}) \;=\; \sum_{i=1}^{n}\sum_{k=1}^{K} r_{ik}\,\|x_i - \mu_k\|^2 ,
\qquad \sum_{k} r_{ik} = 1 .
$$

Đây là hàm của **hai nhóm biến**: nhãn $\mathbf{r}$ và tâm $\boldsymbol{\mu}$. Lloyd chính là **coordinate descent** — tối ưu lần lượt từng nhóm biến trong khi giữ nhóm kia cố định:

- **Bước ASSIGN** giữ $\boldsymbol{\mu}$ cố định, tối ưu theo $\mathbf{r}$. Vì $L$ tách rời theo từng điểm $i$, chọn tối ưu là gán mỗi điểm vào tâm gần nhất: $r_{ik}=1 \iff k=\arg\min_j \|x_i-\mu_j\|^2$. Đây là **nghiệm tối ưu toàn cục** của bài toán con đó $\Rightarrow$ $L$ không thể tăng.
- **Bước UPDATE** giữ $\mathbf{r}$ cố định, tối ưu theo $\boldsymbol{\mu}$. Hàm $\mu_k \mapsto \sum_i r_{ik}\|x_i-\mu_k\|^2$ là hàm bậc hai lồi; đạo hàm bằng 0 cho
$$
\frac{\partial L}{\partial \mu_k} = -2\sum_i r_{ik}(x_i-\mu_k) = 0
\;\Longrightarrow\;
\mu_k = \frac{\sum_i r_{ik} x_i}{\sum_i r_{ik}} = \text{trung bình cụm } k .
$$
Lại là nghiệm tối ưu toàn cục của bài toán con $\Rightarrow$ $L$ không thể tăng.

Vậy dãy $L$ **giảm đơn điệu** và bị **chặn dưới bởi 0**, nên hội tụ. Hơn nữa số cách gán nhãn là **hữu hạn** ($K^n$), mà $L$ không bao giờ tăng, nên thuật toán phải dừng sau hữu hạn bước — không thể chạy mãi.

> **Đây cũng chính là lý do trung bình cộng (mean) xuất hiện trong tên "K-Means"**: nó là nghiệm tối ưu của tổng bình phương khoảng cách. Nếu đổi sang khoảng cách $L_1$ thì nghiệm tối ưu là **trung vị** — ta được thuật toán K-Medians.

![Inertia giảm đơn điệu qua từng nửa bước](images/02_inertia_giam_don_dieu.png)

*Mỗi vòng lặp gồm 2 nửa bước, và cả hai đều kéo inertia xuống. Đường cong phẳng dần rồi nằm ngang — đó là lúc hội tụ. Lưu ý: hội tụ chỉ đảm bảo **cực tiểu địa phương**.*


### 3.3. Bài toán K-Means là NP-hard — nên Lloyd chỉ là heuristic

Đây là chi tiết hay bị bỏ qua nhưng rất quan trọng về mặt tư duy:

- Tìm **nghiệm tối ưu toàn cục** của $\min_{\mathbf{r},\boldsymbol{\mu}} L$ là bài toán **NP-hard**, ngay cả khi $d=2$ (Mahajan et al., 2009) hoặc khi $K=2$ (Aloise et al., 2009). Không ai biết thuật toán đa thức giải đúng nó.
- Lloyd **không phải** thuật toán giải bài toán đó. Nó là một heuristic hội tụ nhanh về một **cực tiểu địa phương**, và cực tiểu ấy có thể tệ hơn nghiệm tối ưu **tuỳ ý** (xem hình ở mục 4.2 phía dưới: 2468.7 so với 375.6, tệ hơn 6.5 lần).
- **`n_init`** trong sklearn chính là cách phòng thủ thô sơ nhưng hiệu quả: chạy toàn bộ thuật toán `n_init` lần với khởi tạo khác nhau, giữ lại lần có inertia thấp nhất. Từ sklearn 1.4, `n_init='auto'` là mặc định (= 1 khi `init='k-means++'`, = 10 khi `init='random'`).

> **Chốt tư duy:** khi bạn gọi `KMeans(...).fit(X)`, bạn KHÔNG nhận được "phân cụm tốt nhất". Bạn nhận được "phân cụm tốt nhất trong `n_init` lần thử ngẫu nhiên". Chạy lại với `random_state` khác có thể ra kết quả khác — và điều đó là **bình thường**, không phải bug.

### 3.4. Độ phức tạp và dữ liệu lớn

Mỗi vòng lặp phải tính khoảng cách từ $n$ điểm tới $K$ tâm trong không gian $d$ chiều:

$$
\mathcal{O}(n \cdot K \cdot d) \text{ mỗi vòng} \quad\Longrightarrow\quad \mathcal{O}(n \cdot K \cdot d \cdot i) \text{ cho } i \text{ vòng lặp}.
$$

Bộ nhớ là $\mathcal{O}(n\,d + K\,d)$ — rất nhẹ (khác hẳn Agglomerative cần ma trận khoảng cách $\mathcal{O}(n^2)$). Đó là lý do K-Means vẫn sống khoẻ với hàng triệu điểm.

| Tình huống | Nên dùng | Vì sao |
|---|---|---|
| $n$ vài nghìn – vài trăm nghìn | `KMeans` | đủ nhanh, chính xác |
| $n$ hàng triệu, cần nhanh | `MiniBatchKMeans` | mỗi bước chỉ lấy 1 mini-batch (vd. 1024 điểm) để cập nhật tâm → nhanh hơn 10–100 lần, inertia chỉ tệ hơn vài % |
| $d$ rất lớn (text TF-IDF vài chục nghìn chiều) | giảm chiều (PCA/SVD) rồi mới K-Means | khoảng cách Euclid mất ý nghĩa ở chiều cao (*curse of dimensionality*) |
| dữ liệu không vừa RAM | `MiniBatchKMeans` + `partial_fit` | học theo luồng |


# THỰC HÀNH 1: K-Means trên dữ liệu giả lập 2D

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons
from sklearn.cluster import KMeans, DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, silhouette_samples

np.random.seed(42)

# Sinh 4 cụm tròn
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.7, random_state=42)

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], s=20, alpha=0.7)
plt.title('Dữ liệu thô (chưa biết nhãn)'); plt.grid(alpha=0.3); plt.show()

In [ ]:
km = KMeans(n_clusters=4, n_init=10, random_state=42)
labels = km.fit_predict(X)

plt.figure(figsize=(7, 5))
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=20, alpha=0.7)
plt.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
            c='red', marker='X', s=200, edgecolor='black', linewidth=2,
            label='Centroids')
plt.title(f'K-Means k=4, inertia = {km.inertia_:.2f}')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 5. Chọn K bằng Elbow + Silhouette

In [ ]:
Ks = list(range(2, 11))
inertias, silhouettes = [], []
for k in Ks:
    km_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit(X)
    inertias.append(km_k.inertia_)
    silhouettes.append(silhouette_score(X, km_k.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(Ks, inertias, 'o-')
axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia (within-cluster SSE)')
axes[0].set_title('Elbow method'); axes[0].grid(alpha=0.3)
axes[0].axvline(4, color='red', linestyle='--', alpha=0.5, label='Khuỷu tay tại K=4')
axes[0].legend()

axes[1].plot(Ks, silhouettes, 'o-', color='green')
axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette score')
axes[1].set_title('Silhouette (cao = tốt)'); axes[1].grid(alpha=0.3)
best_k = Ks[int(np.argmax(silhouettes))]
axes[1].axvline(best_k, color='red', linestyle='--', alpha=0.5,
                label=f'Best K = {best_k}')
axes[1].legend()
plt.tight_layout(); plt.show()

### 5.1. Đọc kỹ hơn về Elbow: khi nào khuỷu tay là ẢO?

Inertia có một tính chất khiến nó **không bao giờ** dùng được để so sánh trực tiếp giữa các $K$ khác nhau:

$$
K \uparrow \;\Longrightarrow\; L(K) \downarrow \quad\text{(luôn luôn)}, \qquad L(n) = 0 .
$$

Với $K=n$ (mỗi điểm một cụm) inertia bằng 0 — "hoàn hảo" mà vô nghĩa. Vì thế elbow **không tìm giá trị nhỏ nhất**, nó tìm **chỗ gãy** của đường cong: điểm mà thêm một cụm nữa không còn "đáng tiền".

![So sánh elbow rõ ràng và elbow mơ hồ](images/03_elbow_ro_va_mo_ho.png)

*Trái: dữ liệu có 4 cụm thật → khuỷu tay ở K=4 rõ như bẻ que. Phải: dữ liệu sinh ngẫu nhiên ĐỀU, hoàn toàn không có cụm → đường cong trơn tru, không có chỗ gãy nào. Nếu vẫn cố chọn K ở đây thì ta đang "phát hiện" ra cấu trúc mà dữ liệu không hề có.*

> **Bẫy:** elbow là phương pháp *bằng mắt*, có tính chủ quan cao. Trên dữ liệu thật (nhiều chiều, nhiều nhiễu) khuỷu tay thường mờ. Đừng dùng elbow một mình — hãy đối chiếu với silhouette và với **kiến thức nghiệp vụ** (bạn thực sự cần bao nhiêu phân khúc khách hàng?).

### 5.2. Silhouette — đọc hiểu công thức, không chỉ đọc con số

Với mỗi điểm $i$:

- $a_i$ = khoảng cách trung bình từ $i$ tới **các điểm khác trong CÙNG cụm** → đo **độ chặt** (càng nhỏ càng tốt).
- $b_i$ = khoảng cách trung bình từ $i$ tới các điểm của **cụm KHÁC gần nhất** → đo **độ tách** (càng lớn càng tốt).

$$
s(i) \;=\; \frac{b_i - a_i}{\max(a_i,\, b_i)} \;\in\; [-1,\, 1] .
$$

| $s(i)$ | Nghĩa |
|---|---|
| gần $+1$ | điểm nằm sâu trong cụm của nó, xa mọi cụm khác — **rất tốt** |
| gần $0$ | điểm nằm ngay trên **ranh giới** giữa hai cụm — mơ hồ |
| âm | điểm **gần cụm khác hơn cụm của chính nó** — nhiều khả năng bị gán sai |

`silhouette_score` = trung bình của mọi $s(i)$. Nhưng **chỉ nhìn số trung bình là mất thông tin**: một cụm rất tốt có thể che lấp một cụm rất tệ. Vì thế nên vẽ **silhouette plot** — sắp xếp $s(i)$ trong từng cụm rồi vẽ dạng "lưỡi dao":

![Silhouette theo K và silhouette plot cho K tốt / K xấu](images/04_silhouette.png)

*(a) Silhouette trung bình có ĐỈNH rõ ở K=4 — dễ đọc hơn "khuỷu tay". (b) Ở K=4 mọi lưỡi dao đều dày, đều vượt đường trung bình. (c) Ở K=7 xuất hiện những lưỡi dao mỏng dính và thấp — dấu hiệu K đang cắt vụn các cụm thật.*

Cách đọc silhouette plot:
- Lưỡi dao nào **ngắn hơn hẳn** các lưỡi khác → cụm đó nhỏ bất thường, có thể là do K quá lớn.
- Lưỡi dao nào **thấp hơn đường trung bình** (nét đứt đỏ) → cụm đó kém chất lượng.
- Có nhiều $s(i) < 0$ → các cụm chồng lấn nghiêm trọng.

**Nhược điểm của silhouette:** độ phức tạp $\mathcal{O}(n^2)$ (phải tính mọi cặp khoảng cách) → với $n$ lớn hãy dùng `sample_size` trong `silhouette_score`. Ngoài ra silhouette cũng *thiên vị cụm cầu*, nên nó thường chấm điểm thấp cho nghiệm ĐÚNG trên dữ liệu dạng moons.

### 5.3. Ba chỉ số nội bộ khác (không cần nhãn thật)

| Chỉ số | Công thức / ý tưởng | Khoảng | Tốt khi |
|---|---|---|---|
| **Silhouette** | $(b-a)/\max(a,b)$ trung bình | $[-1,1]$ | **càng cao** càng tốt |
| **Davies–Bouldin** | trung bình theo cụm của $\max_{j\ne i}\dfrac{\sigma_i+\sigma_j}{d(\mu_i,\mu_j)}$ (độ phân tán trong cụm chia cho khoảng cách giữa tâm) | $[0,\infty)$ | **càng THẤP** càng tốt |
| **Calinski–Harabasz** | $\dfrac{\mathrm{tr}(B_K)}{\mathrm{tr}(W_K)}\cdot\dfrac{n-K}{K-1}$ (phương sai giữa cụm / phương sai trong cụm) | $[0,\infty)$ | **càng cao** càng tốt |

Cả ba đều **giả định cụm lồi, cầu** — chúng đồng loạt "chấm điểm sai" trên dữ liệu moons. Chúng là công cụ tham khảo, không phải trọng tài tối cao.

### 5.4. Khi CÓ nhãn thật: ARI và NMI

Trong lab ta hay có nhãn thật (vì dữ liệu do ta sinh ra) để kiểm chứng thuật toán. Không thể dùng `accuracy` vì **nhãn cụm chỉ là số hiệu tuỳ ý** — cụm "0" của K-Means có thể tương ứng lớp "2" của nhãn thật. Cần chỉ số **bất biến với hoán vị nhãn**:

- **Adjusted Rand Index (ARI)** — đếm số cặp điểm được xếp *cùng nhau/khác nhau* nhất quán giữa hai cách chia, rồi hiệu chỉnh cho ngẫu nhiên:
$$
\mathrm{ARI} = \frac{\mathrm{RI} - \mathbb{E}[\mathrm{RI}]}{\max(\mathrm{RI}) - \mathbb{E}[\mathrm{RI}]} \le 1,
$$
$\mathrm{ARI}=1$ là trùng khớp hoàn hảo, $\approx 0$ là ngang với gán nhãn ngẫu nhiên, giá trị âm nghĩa là còn tệ hơn ngẫu nhiên. Cận dưới thực tế là $-0.5$ chứ không phải $-1$.

- **Normalized Mutual Information (NMI)** — lượng thông tin chung giữa hai cách chia, chuẩn hoá về $[0,1]$:
$$
\mathrm{NMI}(U,V) = \frac{2\,I(U;V)}{H(U)+H(V)} .
$$

> ⚠️ **Nhớ kỹ**: trong bài toán clustering *thật sự*, ta **KHÔNG có nhãn** — nếu có thì đã làm phân loại có giám sát rồi. ARI/NMI chỉ dùng để **kiểm thử thuật toán** trên dữ liệu đã biết đáp án, hoặc khi có một tập nhỏ được gán nhãn tay để thẩm định. Đừng viết pipeline sản xuất phụ thuộc vào ARI.

In [ ]:
# So sánh 5 chỉ số đánh giá clustering trên cùng dữ liệu, quét K từ 2 đến 8
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                             calinski_harabasz_score,
                             adjusted_rand_score, normalized_mutual_info_score)

print(f"{'K':>2} | {'silhouette↑':>11} | {'Davies-B.↓':>10} | {'Calinski-H.↑':>12} | {'ARI↑':>6} | {'NMI↑':>6}")
print('-' * 68)
for k in range(2, 9):
    lab_k = KMeans(n_clusters=k, n_init=10, random_state=42).fit_predict(X)
    print(f'{k:>2} | {silhouette_score(X, lab_k):>11.3f} | '
          f'{davies_bouldin_score(X, lab_k):>10.3f} | '
          f'{calinski_harabasz_score(X, lab_k):>12.1f} | '
          f'{adjusted_rand_score(y_true, lab_k):>6.3f} | '
          f'{normalized_mutual_info_score(y_true, lab_k):>6.3f}')

print('\nBốn chỉ số đầu KHÔNG dùng y_true (chỉ số nội bộ) — dùng được trong thực tế.')
print('ARI/NMI cần y_true — chỉ dùng khi kiểm thử, thực tế hiếm khi có.')


## 6. Cài K-Means từ scratch

Để hiểu rõ thuật toán, tự cài và so với sklearn.

In [ ]:
class MyKMeans:
    def __init__(self, n_clusters=4, max_iter=100, tol=1e-4, random_state=42):
        self.n_clusters = n_clusters
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state

    def fit(self, X):
        rng = np.random.RandomState(self.random_state)
        # Khởi tạo: chọn n_clusters điểm ngẫu nhiên làm tâm
        idx = rng.choice(len(X), self.n_clusters, replace=False)
        centers = X[idx].copy()

        for it in range(self.max_iter):
            # Assign: mỗi điểm gán vào tâm gần nhất
            d = np.sqrt(((X[:, None, :] - centers[None, :, :]) ** 2).sum(axis=2))
            labels = d.argmin(axis=1)

            # Update: tâm = mean của các điểm trong cụm
            new_centers = np.array([X[labels == k].mean(axis=0)
                                     if (labels == k).any() else centers[k]
                                     for k in range(self.n_clusters)])

            shift = np.linalg.norm(new_centers - centers)
            centers = new_centers
            if shift < self.tol:
                break

        self.cluster_centers_ = centers
        self.labels_ = labels
        self.n_iter_ = it + 1
        self.inertia_ = ((X - centers[labels]) ** 2).sum()
        return self

    def predict(self, X):
        d = np.sqrt(((X[:, None, :] - self.cluster_centers_[None, :, :]) ** 2).sum(axis=2))
        return d.argmin(axis=1)

mine = MyKMeans(n_clusters=4).fit(X)
print(f'My KMeans   inertia: {mine.inertia_:.2f}, hội tụ sau {mine.n_iter_} iter')
print(f'sklearn     inertia: {km.inertia_:.2f}')
print()
print('Lưu ý: với random_state=42, khởi tạo NGẪU NHIÊN của MyKMeans rơi vào cực tiểu địa phương')
print('(inertia cao hơn hẳn sklearn, vốn dùng k-means++ và n_init=10). Thuật toán không sai —')
print('đây chính là hiện tượng ở mục 6b bên dưới. Thử đổi random_state để thấy kết quả thay đổi.')

## 6b. Khởi tạo: nửa còn lại của thuật toán

Lloyd hội tụ về **cực tiểu địa phương**, mà cực tiểu nào thì hoàn toàn do **điểm xuất phát** quyết định. Nói cách khác: *chất lượng của K-Means bằng chất lượng của khởi tạo*.

![Khởi tạo kém dẫn tới cực tiểu địa phương](images/05_khoi_tao_kem.png)

*Cùng một bộ dữ liệu, cùng K=4, chỉ khác `random_state`. Seed 3 tìm đúng 4 cụm (inertia 375.6). Seed 16 rơi vào cực tiểu địa phương kinh điển: một cụm thật bị **xẻ đôi** (xanh dương + xanh lá cùng nằm trên một cụm), hai cụm thật bị **gộp làm một** (đỏ) — inertia 2468.7, tệ hơn 6.5 lần. Panel (c): qua 120 seed, `init='random'` chỉ chạm nghiệm tốt 67% số lần, còn `k-means++` đạt 100%.*

### Thuật toán k-means++ (Arthur & Vassilvitskii, 2007)

1. Chọn tâm đầu tiên $\mu_1$ **ngẫu nhiên đều** trong các điểm dữ liệu.
2. Với mỗi điểm $x$, tính $D(x) = \min_{j \le m}\|x - \mu_j\|$ = khoảng cách tới **tâm gần nhất đã chọn**.
3. Chọn tâm tiếp theo $\mu_{m+1}$ **bằng cách bốc thăm** trong các điểm dữ liệu, với xác suất
$$
p(x) \;=\; \frac{D(x)^2}{\sum_{x'} D(x')^2}.
$$
4. Lặp bước 2–3 cho tới khi đủ $K$ tâm, rồi chạy Lloyd như bình thường.

![Xác suất chọn tâm tiếp theo trong k-means++](images/06_kmeanspp_xac_suat.png)

*Sau mỗi tâm được chọn, "vùng tối" (gần các tâm đã có) gần như hết cơ hội, còn các cụm chưa được đại diện thì sáng rực. Nhờ vậy các tâm khởi tạo tự động trải đều — thay vì rơi cả 3 vào cùng một cụm như `init='random'` hay bị.*

**Vì sao lại bình phương $D(x)^2$ mà không phải $D(x)$?**
- Nếu chọn *đúng* điểm xa nhất (deterministic) thì thuật toán sẽ luôn tóm phải **outlier** — điểm xa nhất thường là điểm nhiễu.
- Nếu chọn *đều* thì không có ưu tiên gì, quay về `init='random'`.
- Bình phương là điểm cân bằng: ưu tiên mạnh điểm xa nhưng vẫn là **xác suất**, nên một outlier lẻ loi khó thắng cả một cụm đông đúc ở xa.

**Đảm bảo lý thuyết:** k-means++ là thuật toán $\mathcal{O}(\log K)$-**competitive** — kỳ vọng của inertia thoả

$$
\mathbb{E}[L_{\text{k-means++}}] \;\le\; 8(\ln K + 2)\, L_{\text{OPT}} .
$$

Đây là đảm bảo cho **riêng bước khởi tạo**, trước cả khi chạy Lloyd; chạy Lloyd sau đó chỉ làm inertia giảm tiếp. Đó là điều mà `init='random'` **không có bất kỳ đảm bảo nào** — nó có thể tệ hơn nghiệm tối ưu bao nhiêu lần cũng được.

> Trong sklearn, `KMeans` mặc định đã dùng `init='k-means++'`. Đừng đổi sang `'random'` trừ khi bạn muốn **minh hoạ** đúng vấn đề này.


## 7. Khi K-Means thất bại — cụm cong

Hai trăng lưỡi liềm — K-Means giả định cụm tròn nên không xử lý được.

In [ ]:
X_moon, y_moon = make_moons(n_samples=300, noise=0.08, random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# K-Means
km_moon = KMeans(n_clusters=2, n_init=10, random_state=42).fit(X_moon)
axes[0].scatter(X_moon[:, 0], X_moon[:, 1], c=km_moon.labels_, cmap='coolwarm', s=20)
axes[0].scatter(km_moon.cluster_centers_[:, 0], km_moon.cluster_centers_[:, 1],
                c='black', marker='X', s=200)
axes[0].set_title('K-Means: SAI — cắt ngang hai trăng')
axes[0].grid(alpha=0.3)

# DBSCAN — phát hiện cụm theo mật độ
dbscan = DBSCAN(eps=0.2, min_samples=5).fit(X_moon)
# LƯU Ý: nhãn -1 của DBSCAN nghĩa là NHIỄU, không phải một cụm → phải trừ ra khi đếm
n_clusters = len(set(dbscan.labels_) - {-1})
n_noise = (dbscan.labels_ == -1).sum()
axes[1].scatter(X_moon[:, 0], X_moon[:, 1], c=dbscan.labels_, cmap='coolwarm', s=20)
axes[1].set_title(f'DBSCAN: ĐÚNG — bám theo mật độ '
                  f'({n_clusters} cụm, {n_noise} điểm nhiễu)')
axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('K-Means không xử lý được cụm cong. DBSCAN, Spectral Clustering giải quyết tốt hơn.')

## 7b. Giới hạn HÌNH HỌC của K-Means: vì sao mọi cụm đều lồi

Việc K-Means "chỉ xử lý được cụm tròn" không phải là kinh nghiệm mơ hồ — nó là **hệ quả toán học** của quy tắc gán nhãn.

Điểm $x$ thuộc cụm $i$ khi và chỉ khi $\|x-\mu_i\|^2 \le \|x-\mu_j\|^2$ với mọi $j$. Khai triển:

$$
\|x\|^2 - 2\mu_i^{\top}x + \|\mu_i\|^2 \;\le\; \|x\|^2 - 2\mu_j^{\top}x + \|\mu_j\|^2
\;\Longleftrightarrow\;
2(\mu_j - \mu_i)^{\top} x \;\le\; \|\mu_j\|^2 - \|\mu_i\|^2 .
$$

Số hạng $\|x\|^2$ **triệt tiêu**, còn lại một **bất đẳng thức tuyến tính** theo $x$ — tức là một **nửa không gian**. Cụm $i$ là giao của $K-1$ nửa không gian như vậy, mà giao của các nửa không gian luôn là **tập lồi**. Đây chính là **sơ đồ Voronoi** của tập tâm.

![Ranh giới K-Means là sơ đồ Voronoi, mọi cụm đều lồi](images/08_voronoi_cum_loi.png)

*Biên giữa hai tâm bất kỳ là **đường trung trực** của đoạn nối chúng — một đường thẳng. Vì thế mọi cụm K-Means tìm được đều là đa giác lồi: cụm hình chữ C, hình vành khuyên hay xoắn ốc là **bất khả thi về nguyên tắc**, không phải do "chưa train đủ".*

### Bốn kiểu dữ liệu khiến K-Means gãy

![Bốn trường hợp K-Means thất bại](images/07_kmeans_that_bai.png)

*Bốn chế độ hỏng kinh điển. Mỗi panel ghi rõ nguyên nhân. Điểm chung: K-Means giả định cụm CẦU, kích thước và mật độ tương đương, và gán CỨNG.*

| Triệu chứng | Nguyên nhân gốc | Cách chữa |
|---|---|---|
| Cụm cong, lồng nhau bị cắt ngang | ranh giới bắt buộc là siêu phẳng | DBSCAN, Spectral Clustering, hoặc kernel/embedding trước |
| Cụm hẹp "nuốt" rìa cụm rộng | inertia phạt theo khoảng cách, không mô hình hoá phương sai | **GMM** với `covariance_type` phù hợp |
| Cụm elip xiên bị cắt chéo | K-Means dùng khoảng cách Euclid đẳng hướng | GMM `covariance_type='full'`, hoặc PCA/whitening trước |
| Cụm to bị xẻ đôi, cụm nhỏ bị gộp | inertia là **tổng**, cụm đông đóng góp lớn hơn | DBSCAN; hoặc `sample_weight`; hoặc chấp nhận và hậu xử lý |


## 7c. K-Means trong họ hàng: GMM, K-Medoids, K-Modes

### 7c.1. K-Means là trường hợp GIỚI HẠN của Gaussian Mixture Model

GMM giả định dữ liệu sinh ra từ hỗn hợp $K$ phân phối Gauss và huấn luyện bằng thuật toán **EM**:

- **Bước E** (tương ứng ASSIGN): tính **trách nhiệm mềm** $\gamma_{ik} = P(z_i = k \mid x_i)$ — xác suất điểm $i$ thuộc cụm $k$. Tổng theo $k$ bằng 1, nhưng giá trị nằm trong $(0,1)$ chứ không phải $\{0,1\}$.
- **Bước M** (tương ứng UPDATE): cập nhật $\pi_k, \mu_k, \Sigma_k$ theo trung bình **có trọng số** $\gamma_{ik}$.

Nếu ép mọi hiệp phương sai thành **cầu và bằng nhau**, $\Sigma_k = \sigma^2 I$, rồi cho $\sigma^2 \to 0$, thì

$$
\gamma_{ik} = \frac{\pi_k \exp\!\big(-\|x_i-\mu_k\|^2 / 2\sigma^2\big)}{\sum_j \pi_j \exp\!\big(-\|x_i-\mu_j\|^2/2\sigma^2\big)}
\;\xrightarrow[\sigma^2 \to 0]{}\;
\begin{cases} 1 & k = \arg\min_j \|x_i-\mu_j\| \\ 0 & \text{ngược lại}\end{cases}
$$

— trách nhiệm mềm sụp thành **gán cứng**, và EM trở thành đúng Lloyd. Nói gọn: **K-Means = GMM cầu + gán cứng**. (Vì lý do này K-Means đôi khi được gọi là "hard EM".)

**Khi nào GMM tốt hơn:**
- Cụm có **hình elip / xiên / phương sai khác nhau** → `covariance_type='full'` hoặc `'diag'`.
- Cần **xác suất thuộc cụm** thay vì nhãn cứng (ví dụ: "khách này 70% thuộc phân khúc A, 30% thuộc B").
- Cần **chọn K bằng tiêu chí thống kê**: GMM có likelihood nên dùng được **AIC/BIC** — một cách chọn K nguyên tắc hơn hẳn elbow.

**Cái giá phải trả:** GMM có nhiều tham số hơn ($K d^2/2$ cho full covariance) nên cần nhiều dữ liệu hơn, chậm hơn, và cũng chỉ hội tụ về cực tiểu địa phương (thường được khởi tạo bằng... K-Means).

### 7c.2. K-Medoids (PAM) — khi tâm phải là ĐIỂM DỮ LIỆU THẬT

| | K-Means | K-Medoids (PAM) |
|---|---|---|
| Tâm cụm | **trung bình** — thường không trùng điểm nào có thật | **medoid** = một điểm dữ liệu có thật trong cụm |
| Hàm mục tiêu | $\sum_i d(x_i,\mu)^2$ với $d$ là khoảng cách Euclid | $\sum d(x, m)$ với $d$ là **khoảng cách bất kỳ** |
| Chịu outlier | kém (trung bình bị kéo lệch) | tốt (medoid như trung vị) |
| Độ phức tạp | $\mathcal{O}(nKdi)$ — rẻ | $\mathcal{O}(K(n-K)^2)$ mỗi vòng — đắt |
| Dùng được với | chỉ vector số + Euclid | **ma trận khoảng cách bất kỳ**: cosine, Manhattan, edit distance, DTW... |

K-Medoids là lựa chọn khi bạn chỉ có **ma trận khoảng cách** (ví dụ độ tương đồng chuỗi DNA, khoảng cách giữa các lộ trình) và khi cần một "đại diện có thật" cho mỗi cụm (ví dụ: chọn 5 khách hàng thật làm đại diện 5 phân khúc). Cài đặt: `sklearn_extra.cluster.KMedoids`.

### 7c.3. Dữ liệu phân loại: K-Means KHÔNG dùng được

Với các cột dạng `{"Hà Nội", "Đà Nẵng", "Cần Thơ"}`, người ta hay mã hoá thành `{0, 1, 2}` rồi ném vào K-Means. **Đây là lỗi.** Mã hoá đó tạo ra một thứ tự và một khoảng cách giả: "Hà Nội cách Đà Nẵng 1 đơn vị, cách Cần Thơ 2 đơn vị" là vô nghĩa. Tệ hơn, trung bình của các nhãn (`1.4`) không tương ứng với giá trị nào.

| Loại dữ liệu | Thuật toán | Khoảng cách / tâm |
|---|---|---|
| Toàn số | K-Means | Euclid / trung bình |
| Toàn phân loại | **K-Modes** | số thuộc tính khác nhau (Hamming) / **mode** (giá trị xuất hiện nhiều nhất) |
| Hỗn hợp số + phân loại | **K-Prototypes** | tổ hợp có trọng số của hai loại trên |
| Có ma trận khoảng cách tuỳ ý | K-Medoids, Agglomerative, DBSCAN | do người dùng định nghĩa |

*(One-hot encoding rồi K-Means là một cách chữa cháy tạm được, nhưng nó biến khoảng cách giữa hai giá trị khác nhau bất kỳ thành hằng số $\sqrt{2}$ và làm loãng dữ liệu ở chiều cao — K-Modes vẫn tự nhiên hơn.)*

### 7c.4. Bảng so sánh 4 thuật toán clustering hay dùng nhất

![So sánh K-Means, GMM, Agglomerative, DBSCAN trên 3 bộ dữ liệu khó](images/10_so_sanh_thuat_toan.png)

*Cùng dữ liệu, bốn thuật toán, kết quả khác hẳn nhau. K-Means và Ward đều "thích" cụm cầu nên gãy cả 3 lần; GMM ôm được cụm elip; DBSCAN bám mật độ nên giải được cả moons lẫn vòng lồng nhau — và nó là thuật toán duy nhất biết nói "điểm này là nhiễu".*

| | **K-Means** | **DBSCAN** | **Agglomerative (Ward)** | **GMM** |
|---|---|---|---|---|
| Giả định hình dạng cụm | cầu, kích thước tương đương | **bất kỳ**, miễn mật độ đồng đều | cầu (Ward); linh hoạt hơn với linkage khác | **elip** (tuỳ `covariance_type`) |
| Cần biết trước $K$? | **Có** | **Không** (nhưng phải chọn `eps`, `min_samples`) | Không bắt buộc (cắt dendrogram sau) | **Có** (nhưng chọn được bằng BIC/AIC) |
| Xử lý nhiễu / outlier | Không — mọi điểm đều bị gán | **Có** — gắn nhãn $-1$ | Không | Không (nhưng xác suất thấp là dấu hiệu) |
| Kết quả | nhãn cứng | nhãn cứng + nhiễu | cây phân cấp (dendrogram) | **xác suất mềm** |
| Độ phức tạp thời gian | $\mathcal{O}(nKdi)$ — nhanh nhất | $\mathcal{O}(n\log n)$ có chỉ mục không gian, $\mathcal{O}(n^2)$ không có | $\mathcal{O}(n^2\log n)$ | $\mathcal{O}(nKd^2i)$ |
| Bộ nhớ | $\mathcal{O}(nd)$ | $\mathcal{O}(nd)$ | $\mathcal{O}(n^2)$ — **giới hạn ~10k điểm** | $\mathcal{O}(nd + Kd^2)$ |
| Điểm yếu chí mạng | cụm cong, cụm lệch kích thước | mật độ không đều, khó chỉnh `eps` ở chiều cao | không co giãn được | cần nhiều dữ liệu, dễ suy biến |
| Dự đoán cho điểm MỚI | `predict()` — rẻ | không có `predict` | không có `predict` | `predict_proba()` |


## 8. Ứng dụng thực tế: nén ảnh bằng K-Means

Một ảnh màu 24-bit có thể chứa tới $2^{24} \approx 16.7$ triệu màu khác nhau, mỗi pixel tốn 24 bit. Nếu chỉ giữ $K=16$ màu chính (bằng cách clustering các pixel), mỗi pixel chỉ còn cần **4 bit** để ghi chỉ số màu trong bảng màu (cộng một bảng 16 màu rất nhỏ) → dữ liệu màu giảm khoảng **6 lần** ($24/4$), mà ảnh vẫn nhận ra được.

*(Đừng nhầm: số MÀU giảm cả triệu lần, nhưng KÍCH THƯỚC dữ liệu chỉ giảm theo số bit mỗi pixel. Với $K$ màu thì mỗi pixel cần $\lceil \log_2 K \rceil$ bit.)*

In [ ]:
from sklearn.datasets import load_sample_image

# Tải ảnh sample (đã có trong sklearn)
try:
    img = load_sample_image('china.jpg')
except Exception:
    print('Không tải được ảnh sample, bỏ qua phần này')
    img = None

if img is not None:
    h, w, c = img.shape
    pixels = img.reshape(-1, 3) / 255.0

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    axes[0].imshow(img); axes[0].set_title('Gốc (16M màu)'); axes[0].axis('off')

    for ax, K in zip(axes[1:], [4, 8, 16]):
        # Lấy mẫu để fit nhanh
        sample = pixels[np.random.choice(len(pixels), 5000, replace=False)]
        km = KMeans(n_clusters=K, n_init=3, random_state=42).fit(sample)
        new_pixels = km.cluster_centers_[km.predict(pixels)]
        new_img = new_pixels.reshape(h, w, 3)
        ax.imshow(new_img); ax.set_title(f'K = {K} màu'); ax.axis('off')
    plt.tight_layout(); plt.show()

## 9. Tiền xử lý và những cái bẫy chết người

### 9.1. Vì sao BẮT BUỘC phải scale

K-Means tối ưu tổng bình phương khoảng cách **Euclid**. Khoảng cách Euclid cộng bình phương chênh lệch của mọi feature **theo đơn vị gốc**, nên feature nào có biên độ lớn sẽ chi phối toàn bộ:

$$
\|x - \mu\|^2 = \underbrace{(\text{tuổi} - \mu_{\text{tuổi}})^2}_{\text{chênh } \sim 10^1} + \underbrace{(\text{thu nhập} - \mu_{\text{thu nhập}})^2}_{\text{chênh } \sim 10^7}
$$

Số hạng thứ hai lớn hơn số hạng thứ nhất cỡ $10^{12}$ lần — **hàng nghìn tỷ lần** — feature "tuổi" coi như bị xoá khỏi bài toán.

![Ảnh hưởng của việc quên scale](images/09_vi_sao_phai_scale.png)

*Ba nhóm khách hàng thật (a). Không scale (b): hai nhóm khác tuổi nhưng **cùng thu nhập** bị gộp làm một, còn nhóm thu nhập cao lại bị xẻ đôi theo thu nhập — ARI chỉ 0.44. Sau `StandardScaler` (c): tìm đúng cả ba nhóm, ARI = 1.00.*

| Bộ scale | Khi nào dùng |
|---|---|
| `StandardScaler` | mặc định; feature xấp xỉ chuẩn, không nhiều outlier |
| `MinMaxScaler` | cần giới hạn cứng $[0,1]$; nhưng **rất nhạy outlier** |
| `RobustScaler` | có outlier rõ (dùng trung vị và IQR) |
| `PowerTransformer` / `log1p` | phân phối lệch mạnh (thu nhập, lượt xem, doanh thu) |

> ⚠️ Scale bằng cách **fit trên tập train** rồi `transform` cho dữ liệu mới. Và nhớ: **scale làm thay đổi bài toán** — nó ngầm tuyên bố "mọi feature quan trọng như nhau". Nếu bạn biết một feature quan trọng gấp đôi, hãy nhân trọng số cho nó sau khi scale.

### 9.2. Dữ liệu có thực sự CÓ cụm không? — Hopkins statistic

K-Means **luôn luôn** trả về $K$ cụm, kể cả khi bạn đưa vào nhiễu trắng thuần tuý. Nó không có nút "dữ liệu này không có cấu trúc". Trước khi phân cụm, nên kiểm tra **cluster tendency** bằng **thống kê Hopkins**:

1. Lấy ngẫu nhiên $m$ điểm thật từ dữ liệu; với mỗi điểm, tính $w_i$ = khoảng cách tới **láng giềng gần nhất** trong dữ liệu.
2. Sinh $m$ điểm ngẫu nhiên **đều** trong cùng miền giá trị; với mỗi điểm, tính $u_i$ = khoảng cách tới láng giềng gần nhất trong dữ liệu thật.
3. $\displaystyle H = \frac{\sum u_i}{\sum u_i + \sum w_i}$.

| $H$ | Kết luận |
|---|---|
| $\approx 0.5$ | dữ liệu phân bố **đều** → **KHÔNG có cụm**, đừng phân cụm |
| $> 0.75$ | có xu hướng cụm rõ → phân cụm có ý nghĩa |
| $\approx 0$ | dữ liệu phân bố quá đều đặn (dạng lưới) |

### 9.3. Danh sách bẫy — kiểm tra trước khi nộp bài

| # | Bẫy | Hậu quả | Cách tránh |
|---|---|---|---|
| 1 | **Quên scale** | feature đơn vị lớn nuốt hết phần còn lại | luôn `StandardScaler` trong `Pipeline` |
| 2 | **Dùng inertia thô để so K** | inertia luôn giảm theo $K$ → chọn $K = n$ | dùng elbow (chỗ gãy), silhouette, BIC |
| 3 | **Chọn K theo cảm tính rồi biện minh sau** | cụm vô nghĩa nhưng "nghe hợp lý" | quyết định bằng chỉ số **và** nghiệp vụ, ghi rõ lý do |
| 4 | **Phân cụm dữ liệu không có cấu trúc** | ra "phân khúc" hoàn toàn ngẫu nhiên | kiểm tra Hopkins / vẽ PCA 2D trước |
| 5 | **Mã hoá nhãn phân loại thành 0,1,2 rồi K-Means** | tạo thứ tự và khoảng cách giả | K-Modes / K-Prototypes / one-hot |
| 6 | **Chỉ chạy `n_init=1`** | dễ mắc cực tiểu địa phương | để mặc định hoặc `n_init=10` |
| 7 | **So sánh inertia giữa 2 bộ dữ liệu khác nhau** | inertia phụ thuộc $n$ và thang đo → vô nghĩa | dùng chỉ số chuẩn hoá (silhouette) |
| 8 | **Đặt tên cụm rồi coi như sự thật** | "cụm 2 = khách hàng VIP" chỉ là *diễn giải* | kiểm chứng bằng dữ liệu ngoài / A-B test |
| 9 | **Chạy lại thấy nhãn đổi số rồi tưởng sai** | nhãn cụm là số hiệu tuỳ ý, không có thứ tự | cố định `random_state`, so sánh bằng ARI |
| 10 | **Áp K-Means cho dữ liệu chiều rất cao** | khoảng cách Euclid mất khả năng phân biệt | PCA/UMAP giảm chiều trước, hoặc dùng cosine + Spherical K-Means |


## Tổng kết

1. K-Means: thuật toán **không giám sát**, chia dữ liệu thành $K$ cụm sao cho tổng khoảng cách trong cụm nhỏ nhất.
2. Lloyd's algorithm: lặp Assign → Update đến hội tụ.
3. **Chọn K**: Elbow method và Silhouette score.
4. **K-Means++** (default sklearn) khởi tạo thông minh hơn random.
5. **Phải scale feature** vì dùng Euclidean distance.
6. K-Means *thất bại* với cụm cong → DBSCAN, Spectral Clustering.
7. Ứng dụng: customer segmentation, image quantization, document clustering.

# BÀI TẬP VỀ NHÀ

## Bài 1: Customer segmentation
Tự sinh dataset 500 "khách hàng" với 3 feature: tuổi, thu nhập, số lần mua/tháng. Apply K-Means với K=3,4,5. Dùng silhouette để chọn K tốt nhất. Mô tả từng cụm: tuổi/thu nhập/mua sắm có gì đặc trưng?

*Gợi ý:* `np.random.normal` để sinh, scale trước khi cluster, đặt tên cho cụm như "trẻ–chi nhiều", "trung niên–chi vừa"...

## Bài 2: Ảnh hưởng của khởi tạo
Trên `make_blobs` 4 cụm:
1. Train K-Means với `init='random'`, lặp 10 lần với 10 random_state khác nhau.
2. Train với `init='k-means++'`, cũng 10 lần.
3. Vẽ histogram của inertia. K-means++ có ổn định hơn không?

## Bài 3: Scale có luôn giúp K-Means không?

Làm trên **hai** dataset, mỗi cái hai phiên bản (có `StandardScaler` và không), rồi so với nhãn thật bằng `adjusted_rand_score`:

| Dataset | K | Gợi ý |
|---|---|---|
| `load_wine()` | 3 | các feature lệch thang rất mạnh (proline cỡ hàng nghìn, hue cỡ đơn vị) |
| `load_iris()` | 3 | bốn feature đều là chiều dài tính bằng cm, cùng thang |

1. Với mỗi trường hợp, chạy `KMeans(n_init=10, random_state=...)` vài seed rồi lấy ARI trung bình.
2. Lập bảng 4 dòng kết quả.
3. Trên Wine, scale giúp hay hại? Mức chênh lệch bao nhiêu?
4. **Trên Iris kết quả đi ngược lại dự đoán thông thường.** Hãy tự chạy để thấy, rồi giải thích vì sao.

*Gợi ý cho câu 4:* `StandardScaler` ép **mọi** feature về cùng phương sai 1, kể cả feature gần như không phân biệt được các lớp. Với Iris, hai feature cánh hoa (petal) phân biệt loài rất tốt và vốn đã có phương sai lớn hơn; chuẩn hoá vô tình **hạ trọng số** của chúng xuống ngang với hai feature đài hoa (sepal) vốn nhiễu hơn.

*Bài học rút ra:* "luôn scale trước K-Means" là quy tắc mặc định tốt khi các feature khác đơn vị, nhưng nó **không phải định luật**. Khi mọi feature đã cùng đơn vị và bạn tin rằng feature có phương sai lớn thật sự quan trọng hơn, chuẩn hoá có thể làm hỏng tín hiệu. Hãy đo, đừng tin quy tắc một cách mù quáng.

## Bài 4: Image quantization tự chọn ảnh
Dùng ảnh bất kỳ (load bằng `PIL.Image` hoặc `cv2.imread`). Apply K-Means với K = 2, 4, 8, 16, 32, 64 màu. Lưu các phiên bản và so sánh. Tại K nào ảnh "đủ giống" gốc?

## Bài 5: K-Means++ from scratch
Mở rộng class `MyKMeans` ở trên: thay khởi tạo random bằng K-Means++.

Thuật toán:
1. Chọn ngẫu nhiên 1 điểm làm tâm đầu tiên.
2. Với mỗi điểm còn lại, tính khoảng cách $D(x)$ đến tâm gần nhất hiện có.
3. Chọn điểm tiếp theo với xác suất tỷ lệ với $D(x)^2$ — điểm xa các tâm đã có thì khả năng được chọn cao.
4. Lặp đến đủ K tâm.

So sánh inertia trung bình (qua 10 seed) của random init vs K-Means++ init.